# FNO Plots
All plots in one place: training history, evaluation results, and wave predictions.
Run all cells top to bottom. Saves three PNG files:
- `plots_training.png`
- `plots_eval.png`
- `plots_outputs.png`


## 0. Imports & paths

In [21]:
import sys
!{sys.executable} -m pip install matplotlib

from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Paths (adjust if running from a different directory) ──
PROJECT_DIR     = Path.cwd()
if not (PROJECT_DIR / 'data').is_dir():
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR        = PROJECT_DIR / 'data'
SUBMISSION_DIR  = PROJECT_DIR / 'FNO_submission'
CHECKPOINT_PATH = SUBMISSION_DIR / 'fno_modes64_38traj_seed0.pt'
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXAMPLE_INDEX   = 0   # which eval example to visualise in Section 3
RESOLUTIONS     = [128, 256, 512]

print('Project dir :', PROJECT_DIR)
print('Checkpoint  :', CHECKPOINT_PATH)
print('Device      :', DEVICE)


Project dir : /Users/vedbhatt/Desktop/aml-group2-main/models
Checkpoint  : /Users/vedbhatt/Desktop/aml-group2-main/models/FNO_submission/fno_modes64_38traj_seed0.pt
Device      : cpu


## 1. Model definition & load checkpoint

In [22]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        self.modes = modes
        scale = 1 / (in_channels * out_channels)
        self.weights = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes, dtype=torch.cfloat))

    def forward(self, x):
        batch, _, n_points = x.shape
        x_freq = torch.fft.rfft(x)
        out_freq = torch.zeros(batch, self.weights.shape[1], n_points // 2 + 1,
                               dtype=torch.cfloat, device=x.device)
        k = min(self.modes, n_points // 2 + 1)
        out_freq[:, :, :k] = torch.einsum('bix,iox->box',
                                           x_freq[:, :, :k], self.weights[:, :, :k])
        return torch.fft.irfft(out_freq, n=n_points)


class FNO1d(nn.Module):
    def __init__(self, modes=64, width=64, n_layers=4):
        super().__init__()
        self.lift = nn.Linear(2, width)
        self.spec = nn.ModuleList([SpectralConv1d(width, width, modes) for _ in range(n_layers)])
        self.w    = nn.ModuleList([nn.Conv1d(width, width, 1) for _ in range(n_layers)])
        self.proj = nn.Sequential(nn.Linear(width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, x):
        x = self.lift(x).permute(0, 2, 1)
        for s, w in zip(self.spec, self.w):
            x = nn.functional.gelu(s(x) + w(x))
        return self.proj(x.permute(0, 2, 1)).squeeze(-1)


def make_input(u):
    B, N = u.shape
    x = torch.linspace(0, 1, N + 1, device=u.device)[:-1].expand(B, N)
    return torch.stack([u, x], dim=-1)


ckpt  = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model = FNO1d(**ckpt['config']).to(DEVICE)
model.load_state_dict(ckpt['state'])
model.eval()
print(f"Loaded checkpoint (best epoch: {ckpt['best_epoch']})")


Loaded checkpoint (best epoch: 287)


## 2. Training history
Train and validation relative L² loss over epochs. Saved to `plots_training.png`.

In [23]:
train = np.asarray(ckpt['train_hist'], dtype=float)
val   = np.asarray(ckpt['val_hist'],   dtype=float)
epochs = np.arange(len(train))
best_epoch = ckpt['best_epoch']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: full curves
axes[0].plot(epochs, train, color='C0', linewidth=1.5, label='Train')
axes[0].plot(epochs, val,   color='C1', linewidth=1.5, label='Val')
axes[0].axvline(best_epoch, color='red', linestyle=':', linewidth=1,
               label=f'Best epoch {best_epoch}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Relative L² error')
axes[0].set_yscale('log')
axes[0].set_title('FNO training & validation curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: last 100 epochs
tail = max(0, len(train) - 100)
axes[1].plot(epochs[tail:], train[tail:], color='C0', linewidth=1.5, label='Train')
axes[1].plot(epochs[tail:], val[tail:],   color='C1', linewidth=1.5, label='Val')
axes[1].axvline(best_epoch, color='red', linestyle=':', linewidth=1,
               label=f'Best epoch {best_epoch}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Relative L² error')
axes[1].set_yscale('log')
axes[1].set_title('Final 100 epochs (zoomed)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
out = SUBMISSION_DIR / 'plots_training.png'
plt.savefig(out, dpi=150)
print(f"Saved {out}")
plt.show()


Saved /Users/vedbhatt/Desktop/aml-group2-main/models/FNO_submission/plots_training.png


/var/folders/zp/bmr2z42j0v19n5r231_h7gfh0000gn/T/ipykernel_41222/4265734309.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Evaluation: FNO vs Identity baseline across resolutions
Loads `fno_errors.npz`. Saved to `plots_eval.png`.

In [24]:
errors_path = SUBMISSION_DIR / 'fno_errors.npz'
with np.load(errors_path) as err:
    fno_means      = [err[f'err_{r}'].mean() for r in RESOLUTIONS]
    identity_means = [err[f'id_{r}'].mean()  for r in RESOLUTIONS]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: FNO vs identity bar chart
x, w = np.arange(len(RESOLUTIONS)), 0.35
axes[0].bar(x - w/2, fno_means,      w, label='FNO',      color='C0')
axes[0].bar(x + w/2, identity_means, w, label='Identity', color='C2')
axes[0].set_xticks(x)
axes[0].set_xticklabels([str(r) for r in RESOLUTIONS])
axes[0].set_xlabel('Resolution (grid points)')
axes[0].set_ylabel('Mean relative L² error')
axes[0].set_yscale('log')
axes[0].set_title('FNO vs Identity baseline')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: error growth relative to 128
growth = [fno_means[1] / fno_means[0], fno_means[2] / fno_means[0]]
axes[1].bar(['128→256', '128→512'], growth, color=['C2', 'C3'])
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1)
axes[1].set_ylabel('Error growth ratio')
axes[1].set_title('FNO error growth relative to 128-point resolution')
for i, v in enumerate(growth):
    axes[1].text(i, v, f' {v:.2f}×', ha='center', va='bottom')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
out = SUBMISSION_DIR / 'plots_eval.png'
plt.savefig(out, dpi=150)
print(f"Saved {out}")
plt.show()


Saved /Users/vedbhatt/Desktop/aml-group2-main/models/FNO_submission/plots_eval.png


/var/folders/zp/bmr2z42j0v19n5r231_h7gfh0000gn/T/ipykernel_41222/1046487129.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Wave predictions at 128 / 256 / 512 points
Visualises one evaluation example at each resolution. Saved to `plots_outputs.png`.
Change `EXAMPLE_INDEX` at the top to visualise a different example.

In [25]:
predictions = {}
with torch.no_grad():
    for res in RESOLUTIONS:
        with np.load(f"/Users/vedbhatt/Desktop/aml-group2-main/data/eval_{res}.npz") as d:
            fields      = d['field'].astype(np.float32)
            targets     = d['target'].astype(np.float32)
            coordinates = d['coordinates']

        field_t = torch.from_numpy(fields[EXAMPLE_INDEX:EXAMPLE_INDEX+1]).to(DEVICE)
        pred    = model(make_input(field_t)).cpu().numpy()[0]
        target  = targets[EXAMPLE_INDEX]
        err     = np.linalg.norm(pred - target) / (np.linalg.norm(target) + 1e-8)
        predictions[res] = (coordinates, fields[EXAMPLE_INDEX], target, pred, err)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, res in zip(axes, RESOLUTIONS):
    x, field, target, pred, err = predictions[res]
    ax.plot(x, field,  color='0.6', linestyle=':',  linewidth=1.5, label='Input')
    ax.plot(x, target, color='black', linewidth=2,               label='True next step')
    ax.plot(x, pred,   color='C0', linestyle='--', linewidth=1.5, label='FNO prediction')
    ax.set_title(f'{res} points\nrel L² = {err:.4f}')
    ax.set_xlabel('Position x')
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Solution value')
axes[-1].legend()
fig.suptitle(f'FNO predictions vs true solution (example {EXAMPLE_INDEX})')
fig.tight_layout()
out = SUBMISSION_DIR / 'plots_outputs.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f"Saved {out}")
plt.show()


Saved /Users/vedbhatt/Desktop/aml-group2-main/models/FNO_submission/plots_outputs.png


/var/folders/zp/bmr2z42j0v19n5r231_h7gfh0000gn/T/ipykernel_41222/100443283.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
